In [3]:
# Célula 1: Importações
import sys
import os
import pathlib
import pandas as pd
import numpy as np
import yaml

In [4]:
# Detecção automática de caminhos
notebook_path = pathlib.Path().absolute()
root_folder = notebook_path.parent

# Definir caminhos
SRC_DIR = root_folder / "src/data_processing"
DATA_DIR = root_folder / "data"
COMPLETE_SERIES_DIR = DATA_DIR / "complete_series"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = root_folder / "config"

# Adicionar src ao path
sys.path.append(str(SRC_DIR))

# Importar módulo de feature engineering
from features_processing import process_features

In [3]:
# Carregar configurações do arquivo YAML
config_path = CONFIG_DIR / "data_config.yaml"

try:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    feature_config = config.get('feature_windows', {})
    
    print("✅ Configurações carregadas do arquivo YAML:")
    for key, value in feature_config.items():
        print(f"  - {key}: {value}")
    
    # Extrair configurações
    api_k_list = feature_config.get('api_k_list')
    precipitation_ma_windows = feature_config.get('precipitation_ma')
    precipitation_cumulative_windows = feature_config.get('precipitation_cum')
    forecast_ma_windows = feature_config.get('forecast_ma', precipitation_ma_windows)
    forecast_cumulative_windows = feature_config.get('forecast_cum', precipitation_cumulative_windows)
    evapotranspiration_ma_windows = feature_config.get('evapotranspiration_ma')
    anomaly_ma_windows = feature_config.get('anomaly_ma')
    
except FileNotFoundError:
    print("⚠️  Arquivo de configuração não encontrado. Usando valores padrão.")
    
    # Valores padrão
    api_k_list = [0.70, 0.80, 0.85, 0.90, 0.92, 0.95]
    precipitation_ma_windows = [3, 7, 15]
    precipitation_cumulative_windows = [3, 5, 7, 10]
    forecast_ma_windows = [3, 7, 15]
    forecast_cumulative_windows = [3, 5, 7, 10]
    evapotranspiration_ma_windows = [7, 14, 30]
    anomaly_ma_windows = [3, 7]

✅ Configurações carregadas do arquivo YAML:
  - precipitation_ma: [3, 7, 15]
  - precipitation_cum: [3, 5, 7, 10]
  - evapotranspiration_ma: [7, 14, 30]
  - anomaly_ma: [3, 7]


In [8]:
# Carregar configurações do arquivo YAML
config_path = CONFIG_DIR / "data_config.yaml"

try:
    # Usar o config_loader para carregar e validar
    sys.path.insert(0, str(root_folder / "src/utils"))
    from config_loader import load_feature_config
    
    feature_config = load_feature_config(validate=True)
    
    print("✅ Configurações carregadas do arquivo YAML:")
    print(f"Arquivo: {config_path}")
    print()
    
    for key, value in feature_config.items():
        print(f"  - {key}: {value}")
    # Extrair configurações
    api_k_list = feature_config['api_k_list']
    precipitation_ma_windows = feature_config['precipitation_ma']
    precipitation_cumulative_windows = feature_config['precipitation_cum']
    forecast_ma_windows = feature_config.get('forecast_ma', precipitation_ma_windows)
    forecast_cumulative_windows = feature_config.get('forecast_cum', precipitation_cumulative_windows)
    evapotranspiration_ma_windows = feature_config['evapotranspiration_ma']
    anomaly_ma_windows = feature_config['anomaly_ma']
    
except FileNotFoundError as e:
    print(f"❌ ERRO CRÍTICO: {e}")
    print("\nSolução:")
    print("1. Crie o arquivo data_config.yaml na pasta config/")
    print("2. Adicione a seção 'feature_windows' com as chaves obrigatórias")
    print("3. Execute novamente o notebook")
    
    # Sugerir criação do arquivo
    print("\nPara criar um arquivo de configuração padrão, execute:")
    print(f"  from config_loader import ConfigLoader")
    print(f"  ConfigLoader.create_default_config(pathlib.Path('config'))")
    
    # Parar a execução
    raise
    
except ValueError as e:
    print(f"❌ ERRO DE CONFIGURAÇÃO: {e}")
    print("\nSolução:")
    print(f"1. Edite o arquivo: {config_path}")
    print("2. Corrija as configurações conforme indicado acima")
    print("3. Execute novamente o notebook")
    raise
    
except ImportError as e:
    print(f"❌ ERRO DE IMPORT: {e}")
    print("\nSolução:")
    print("1. Verifique se o módulo config_loader existe em src/utils/")
    print("2. Verifique se o arquivo data_config.yaml existe em config/")
    raise

⚠️  Aviso: 'forecast_ma' não definido. Usando valores de 'precipitation_ma'
⚠️  Aviso: 'forecast_cum' não definido. Usando valores de 'precipitation_cum'
✅ Configurações carregadas do arquivo YAML:
Arquivo: c:\Users\emily\Documents\GitHub\Mini-curso-GitHub-Leo\config\data_config.yaml

  - api_k_list: [0.7, 0.8, 0.85, 0.9, 0.92, 0.95]
  - precipitation_ma: [3, 7, 15]
  - precipitation_cum: [3, 5, 7, 10]
  - evapotranspiration_ma: [7, 14, 30]
  - anomaly_ma: [3, 7]
  - forecast_ma: [3, 7, 15]
  - forecast_cum: [3, 5, 7, 10]


In [9]:
# Configuração das Estações
station_ids = [10100000, 13150000, 14100000]

# Data de corte para estatísticas
train_date_cutoff = "2010-12-31" # TAMBEM PEGAR DE ARQUIVO YAMAL

print("\nConfigurações de janelas por tipo de feature:")
print(f"Precipitação (observada):")
print(f"  - Médias móveis: {precipitation_ma_windows}")
print(f"  - Acumulados: {precipitation_cumulative_windows}")
print(f"\nForecast de precipitação:")
print(f"  - Médias móveis: {forecast_ma_windows}")
print(f"  - Acumulados: {forecast_cumulative_windows}")
print(f"\nEvapotranspiração:")
print(f"  - Médias móveis: {evapotranspiration_ma_windows}")
print(f"\nAnomalias:")
print(f"  - Médias móveis: {anomaly_ma_windows}")
print(f"\nAPI:")
print(f"  - Valores k: {api_k_list}")


# Chamar função de processamento
combined_df = process_features(
    input_dir=COMPLETE_SERIES_DIR,
    output_dir=PROCESSED_DIR,
    station_ids=station_ids,
    # Não precisamos passar parâmetros individuais, serão carregados do config
    api_k_list=None,  # Será carregado do config
    precipitation_ma_windows=None,  # Será carregado do config
    precipitation_cumulative_windows=None,  # Será carregado do config
    forecast_ma_windows=None,  # Será carregado do config
    forecast_cumulative_windows=None,  # Será carregado do config
    evapotranspiration_ma_windows=None,  # Será carregado do config
    anomaly_ma_windows=None,  # Será carregado do config
    train_date_cutoff=train_date_cutoff,
    output_filename="features_combined.csv",
    use_config_file=True  # Habilitar carregamento do arquivo de configuração
)


Configurações de janelas por tipo de feature:
Precipitação (observada):
  - Médias móveis: [3, 7, 15]
  - Acumulados: [3, 5, 7, 10]

Forecast de precipitação:
  - Médias móveis: [3, 7, 15]
  - Acumulados: [3, 5, 7, 10]

Evapotranspiração:
  - Médias móveis: [7, 14, 30]

Anomalias:
  - Médias móveis: [3, 7]

API:
  - Valores k: [0.7, 0.8, 0.85, 0.9, 0.92, 0.95]
Iniciando processamento de features...
⚠️  Aviso: 'forecast_ma' não definido. Usando valores de 'precipitation_ma'
⚠️  Aviso: 'forecast_cum' não definido. Usando valores de 'precipitation_cum'
✅ Configurações carregadas do arquivo YAML
  - api_k_list carregado do config: [0.7, 0.8, 0.85, 0.9, 0.92, 0.95]
  - precipitation_ma carregado do config: [3, 7, 15]
  - precipitation_cum carregado do config: [3, 5, 7, 10]
  - forecast_ma carregado do config: [3, 7, 15]
  - forecast_cum carregado do config: [3, 5, 7, 10]
  - evapotranspiration_ma carregado do config: [7, 14, 30]
  - anomaly_ma carregado do config: [3, 7]

Carregando 3 esta

In [ ]:
'''
#VERSÃO ANTERIOR

# Configuração das Janelas
station_ids = [10100000, 13150000, 14100000]

# Configurações específicas por tipo de feature
api_k_list = [0.70, 0.80, 0.85, 0.90, 0.92, 0.95]

# Janelas para precipitação OBSERVADA
precipitation_ma_windows = [3, 7, 15]          # Médias móveis
precipitation_cumulative_windows = [3, 5, 7, 10]  # Acumulados

# Janelas para FORECAST de precipitação (pode ser diferente)
forecast_ma_windows = [3, 7, 15]
forecast_cumulative_windows = [3, 5, 7, 10]

# Janelas para evapotranspiração
evapotranspiration_ma_windows = [7, 14, 30]

# Janelas para anomalias
anomaly_ma_windows = [3, 7]

# Data de corte para estatísticas
train_date_cutoff = "2010-12-31"

print("Configurações de janelas por tipo de feature:")
print(f"Precipitação (observada):")
print(f"  - Médias móveis: {precipitation_ma_windows}")
print(f"  - Acumulados: {precipitation_cumulative_windows}")
print(f"\nForecast de precipitação:")
print(f"  - Médias móveis: {forecast_ma_windows}")
print(f"  - Acumulados: {forecast_cumulative_windows}")
print(f"\nEvapotranspiração:")
print(f"  - Médias móveis: {evapotranspiration_ma_windows}")
print(f"\nAnomalias:")
print(f"  - Médias móveis: {anomaly_ma_windows}")
print(f"\nAPI:")
print(f"  - Valores k: {api_k_list}")

# Processar features
try:
    features_df = process_features(
        input_dir=COMPLETE_SERIES_DIR,
        output_dir=PROCESSED_DIR,
        station_ids=station_ids,
        api_k_list=api_k_list,
        precipitation_ma_windows=precipitation_ma_windows,
        precipitation_cumulative_windows=precipitation_cumulative_windows,
        forecast_ma_windows=forecast_ma_windows,
        forecast_cumulative_windows=forecast_cumulative_windows,
        evapotranspiration_ma_windows=evapotranspiration_ma_windows,
        anomaly_ma_windows=anomaly_ma_windows,
        train_date_cutoff=train_date_cutoff,
        output_filename="features_combined.csv"
    )
    
    print("\nProcessamento concluído com sucesso!")
    
except Exception as e:
    print(f"\nErro durante o processamento: {e}")
    import traceback
    traceback.print_exc()'''

Configurações de janelas por tipo de feature:
Precipitação (observada):
  - Médias móveis: [3, 7, 15]
  - Acumulados: [3, 5, 7, 10]

Forecast de precipitação:
  - Médias móveis: [3, 7, 15]
  - Acumulados: [3, 5, 7, 10]

Evapotranspiração:
  - Médias móveis: [7, 14, 30]

Anomalias:
  - Médias móveis: [3, 7]

API:
  - Valores k: [0.7, 0.8, 0.85, 0.9, 0.92, 0.95]


In [ ]:
'''# Exemplo de configuração alternativa (mais simples)
# Usar as mesmas janelas para tudo, pode fazer assim:
simple_ma_windows = [3, 7, 15]
simple_cum_windows = [3, 5, 7, 10]

print("\nConfiguração simplificada (mesmas janelas para tudo):")
try:
    features_df_simple = process_features(
        input_dir=COMPLETE_SERIES_DIR,
        output_dir=PROCESSED_DIR,
        station_ids=station_ids,
        api_k_list=api_k_list,
        precipitation_ma_windows=simple_ma_windows,
        precipitation_cumulative_windows=simple_cum_windows,
        forecast_ma_windows=simple_ma_windows,  # Mesmo que precipitação
        forecast_cumulative_windows=simple_cum_windows,  # Mesmo que precipitação
        evapotranspiration_ma_windows=simple_ma_windows,  # Mesmo que precipitação
        anomaly_ma_windows=[3, 7],  # Específico para anomalias
        train_date_cutoff=train_date_cutoff,
        output_filename="features_simple.csv"
    )
    
    print("Configuração simplificada concluída!")
    
except Exception as e:
    print(f"Erro na configuração simplificada: {e}")'''

'# Exemplo de configuração alternativa (mais simples)\n# Se quiser usar as mesmas janelas para tudo, pode fazer assim:\nsimple_ma_windows = [3, 7, 15]\nsimple_cum_windows = [3, 5, 7, 10]\n\nprint("\nConfiguração simplificada (mesmas janelas para tudo):")\ntry:\n    features_df_simple = process_features(\n        input_dir=COMPLETE_SERIES_DIR,\n        output_dir=PROCESSED_DIR,\n        station_ids=station_ids,\n        api_k_list=api_k_list,\n        precipitation_ma_windows=simple_ma_windows,\n        precipitation_cumulative_windows=simple_cum_windows,\n        forecast_ma_windows=simple_ma_windows,  # Mesmo que precipitação\n        forecast_cumulative_windows=simple_cum_windows,  # Mesmo que precipitação\n        evapotranspiration_ma_windows=simple_ma_windows,  # Mesmo que precipitação\n        anomaly_ma_windows=[3, 7],  # Específico para anomalias\n        train_date_cutoff=train_date_cutoff,\n        output_filename="features_simple.csv"\n    )\n\n    print("Configuração simpli